In [8]:
# — Bronze Ingestion ————————————————————————
# Notebook  : nb_bronze_ingestion
# Layer     : Raw → Bronze
# Purpose   : Land raw CSVs as Delta tables, no cleaning,
#             add _source_file + _ingested_at audit columns
# Author    : Charan Gnanappan
# Date      : 2026-06-23

# Cell 1 — Imports and configuration
from pyspark.sql.functions import input_file_name, current_timestamp
from pyspark.sql.types import StructType

BRONZE_DB = "lh_northwind_erp"
RAW_BASE  = "Files/raw"

# Cell 2 — Ingestion function for a single table
def ingest_table(table_name):
    
    raw_path = f"{RAW_BASE}/{table_name}"
    
    # Step 1: Read raw CSV
    df_raw = (spark.read
                   .option("header", "true")
                   .option("inferSchema", "true")
                   .csv(raw_path))
    
    # Step 2: Add audit columns
    df_bronze = (df_raw
                    .withColumn("_source_file", input_file_name())
                    .withColumn("_ingested_at", current_timestamp()))
    
    # Step 3: Row count reconciliation
    raw_count    = df_raw.count()
    bronze_count = df_bronze.count()
    
    if raw_count != bronze_count:
        raise ValueError(
            f"[{table_name}] Row mismatch! "
            f"Raw={raw_count}, Bronze={bronze_count}"
        )
    
    # Step 4: Write to Delta table
    df_bronze.write.format("delta").mode("overwrite").saveAsTable(table_name)
    
    print(f"[{table_name}] Ingested {bronze_count} rows successfully")

# Cell 3 — Run ingestion for all 8 tables
tables = [
    "categories",
    "customers",
    "employees",
    "order_details",
    "orders",
    "products",
    "shippers",
    "suppliers"
]

for table in tables:
    try:
        ingest_table(table)
    except Exception as e:
        print(f"[{table}] FAILED: {e}")

StatementMeta(, cabea2ff-120c-4ee9-9ee1-42193b03f21f, 10, Finished, Available, Finished, False)

[categories] Ingested 8 rows successfully
[customers] Ingested 91 rows successfully
[employees] Ingested 9 rows successfully
[order_details] Ingested 2155 rows successfully
[orders] Ingested 830 rows successfully
[products] Ingested 77 rows successfully
[shippers] Ingested 3 rows successfully
[suppliers] Ingested 29 rows successfully
